# FM steel 3D: hierarchical microstructure from a Voronoi grain structure

This notebook builds a 3D ferritic-martensitic steel microstructure and writes it out as an Abaqus `.inp` deck. It starts from a Voronoi tessellation and then builds the usual hierarchy on top of it:

**grains → prior-austenite grains (PAGs) → packets → blocks → (optional) sub-blocks**

Every block gets a Kurdjumov-Sachs (KS) variant orientation, and each PAG gets its own parent orientation. At the end you get plots of the structure, an optional export of the raw arrays, and the Abaqus deck.

It needs UPXO 1.4.0 or later (`pip install upxo`). Voxel decks don't need the meshing extras.

Each stage has a small block of settings at the top of its cell, with the default values already in place. Change the ones you care about and run the cells in order. If you change a setting, re-run its cell and everything after it.

Author: Dr. Sunil Anandatheertha

## Setup

Imports, the output folder and a report session. The report collects the tables and figures from the run into one `report.html` file, written at the very end.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from upxo.pxtal.fm_steel_3d.steps import (
    steps_start, steps_base_grain_structure, steps_cleaning, steps_transformations,
    steps_pag_clustering, steps_block_generation, steps_subblock_generation,
    steps_visualization, steps_raw_export,
)
from upxo.pxtal.fm_steel_3d.deck_export import export_deck
from upxo.pxtal.fm_steel_3d.visualization_maps import build_visualization_data

In [ ]:
# Everything this notebook writes goes under <OUTPUT_DIR>/FMSteel3D/...
OUTPUT_DIR = steps_start.DEFAULT_OUTPUT_DIR      # or Path('/your/output/folder')

RESEARCHER_NAME = ""
ORGANISATION = ""
EMAIL = ""
MATERIAL_NAME = "Low-alloy steel"
PROCESSING_CONDITION = ""

report = steps_start.open_pipeline_report(
    output_dir=OUTPUT_DIR, researcher_name=RESEARCHER_NAME, organisation=ORGANISATION,
    email=EMAIL, material_name=MATERIAL_NAME, processing_condition=PROCESSING_CONDITION,
    notebook_tag='fm_steel_3d_voronoi')

## 1. Base grain structure (Voronoi)

A Voronoi tessellation of random seed points on a voxel grid. Each seed becomes one grain, so `N_SEEDS` is roughly the number of grains you get. The seeds are placed at random, and `VORONOI_SEED` makes the placement repeatable.

The grid is `NX x NY x NZ` voxels and each voxel is `VOXEL_SIZE` long, so the physical size of the domain is the number of voxels times `VOXEL_SIZE`. `CONNECTIVITY` decides which voxels count as touching when grains are identified: 6 means shared faces only, 26 also counts edges and corners.

In [ ]:
NX, NY, NZ = 50, 50, 50     # voxels along x, y, z
VOXEL_SIZE = 1.0            # edge length of one voxel
UNITS = 'microns'           # unit of VOXEL_SIZE
N_SEEDS = 100               # number of Voronoi cells
VORONOI_SEED = 0            # seed for the random point placement
CONNECTIVITY = 6            # 6 or 26
MIN_GRAIN_NVOXELS = -1      # -1 keeps every grain

lfi = steps_base_grain_structure.generate_voronoi_lfi(NX, NY, NZ, N_SEEDS, VORONOI_SEED)

fm_base = steps_base_grain_structure.build_base(
    lfi, physical_dimensions=(NX * VOXEL_SIZE, NY * VOXEL_SIZE, NZ * VOXEL_SIZE),
    voxel_size=VOXEL_SIZE, units=UNITS, connectivity=CONNECTIVITY,
    min_grain_nvoxels=MIN_GRAIN_NVOXELS, random_seed=VORONOI_SEED or 42)
print(f'{fm_base.n_grains} grains on a {NX} x {NY} x {NZ} grid')

## 2. Cleaning

Merges grains smaller than `CLEAN_THRESHOLD` voxels into a neighbour and removes spike voxels (single voxels hanging off a grain). With a Voronoi structure there is usually little to clean, but it's cheap and keeps the later stages tidy.

In [ ]:
CLEAN_THRESHOLD = 8          # grains below this many voxels are merged away
CLEAN_MAX_PASSES = 100
REMOVE_SPIKES = True

fm_base, clean_stats = steps_cleaning.clean(
    fm_base, threshold=CLEAN_THRESHOLD, max_passes=CLEAN_MAX_PASSES,
    do_remove_spikes=REMOVE_SPIKES)
report.add_table([clean_stats], title='Grain statistics after cleaning', section='Cleaning')
clean_stats

## 3. Transformations (optional)

Off by default. When switched on, `SCALE_FACTOR` resamples the whole grid (2.0 doubles the voxels along each axis and halves the voxel size, so the physical size stays the same). `SF_X`, `SF_Y` and `SF_Z` then stretch the structure along one axis at a time, which makes the grains non-equiaxed. `TRANSFORM_CLEANUP_THRESHOLD` merges away grains that end up smaller than that many voxels.

In [ ]:
APPLY_TRANSFORM = False
SCALE_FACTOR = 1.0
SF_X, SF_Y, SF_Z = 1.0, 1.0, 1.0
TRANSFORM_CLEANUP_THRESHOLD = 0

if APPLY_TRANSFORM:
    fm_base = steps_transformations.apply_transform(
        fm_base, scale_factor=SCALE_FACTOR, sf_x=SF_X, sf_y=SF_Y, sf_z=SF_Z,
        cleanup_threshold=TRANSFORM_CLEANUP_THRESHOLD)
    print(f'{fm_base.n_grains} grains, voxel size {fm_base.voxel_size:g} {fm_base.units}')
else:
    print('Transformations skipped.')

## 4. PAGs and packets

Groups neighbouring grains into prior-austenite grains.

- `PAG_TECHNIQUE`: `'A'` groups neighbouring grains into PAGs. `'B'` goes the other way: every grain becomes a PAG and is split into packets (grains smaller than 32 voxels stay as a single packet).
- `PAG_LEVER`: how technique A handles awkward cases, such as grains left with no neighbour to join. It is ignored by technique B. The options are `'lever1'`, `'lever2'`, `'lever3'` (the default), `'lever1_3'` and `'lever2_3'`.
- `PAG_SIZE_DISTRIBUTION`: how big a PAG should be (`sizes`) and how likely each size is (`probs`, which should add up to 1). A PAG holds at most `MAX_PACKETS_PER_PAG` packets, so bigger sizes are clipped down to that.
- `PAG_GRAIN_FRACTION`: the share of the grain volume that goes into PAGs. 1.0 puts everything into PAGs. Whatever is left over (1 minus this value) is kept as isolated retained-austenite grains.
- `MAX_PACKETS_PER_PAG`: upper limit on the number of packets inside one PAG. The default of 4 matches the four {111} habit planes of FCC austenite.

In [ ]:
PAG_TECHNIQUE = 'A'                       # 'A' or 'B'
PAG_LEVER = 'lever3'                      # technique A only
PAG_SIZE_DISTRIBUTION = {'sizes': [2, 3, 4], 'probs': [0.25, 0.5, 0.25]}
PAG_GRAIN_FRACTION = 1.0
MAX_PACKETS_PER_PAG = 4
PAG_SEED = 42

fm_pag = steps_pag_clustering.generate_pags(
    fm_base, technique=PAG_TECHNIQUE, pag_size_distribution=PAG_SIZE_DISTRIBUTION,
    max_packets_per_pag=MAX_PACKETS_PER_PAG, lever=PAG_LEVER,
    pag_grain_fraction=PAG_GRAIN_FRACTION, random_seed=PAG_SEED)
print(f'{len(fm_pag.clusters_dict)} PAGs, {len(fm_pag.retained_austenite_pag_ids)} of them retained austenite')

## 5. PAG orientations

Gives every PAG a parent (austenite) orientation, as Bunge Euler angles.

- `'random'`: uniformly random orientations.
- `'hagb_constrained'`: random, but neighbouring PAGs must differ by at least `hagb_threshold` degrees (a high-angle boundary). Extra settings go in `PAG_ORI_PARAMS`: `hagb_threshold` (default 15), `max_attempts` (default 1000) and an optional `orientation_pool`.
- `'textured'`: orientations are drawn from a synthetic texture. You must give `tc_info` in `PAG_ORI_PARAMS`, a dictionary of texture components, each as `[volume fraction, [phi1, Phi, phi2]]`. Optionally `N` (pool size) and `hagb_constraint`.
- `'fixed'`: every PAG gets the same orientation, given as `euler_angles=(phi1, Phi, phi2)` in `PAG_ORI_PARAMS`.

In [ ]:
PAG_ORI_MODE = 'random'      # 'random', 'hagb_constrained', 'textured' or 'fixed'
PAG_ORI_PARAMS = None        # e.g. {'hagb_threshold': 15.0} or {'euler_angles': (0.0, 0.0, 0.0)}
PAG_ORI_SEED = 42

steps_pag_clustering.assign_pag_orientations(
    fm_pag, pag_ori_mode=PAG_ORI_MODE, pag_ori_params=PAG_ORI_PARAMS, random_seed=PAG_ORI_SEED)
print(f'{len(fm_pag.pag_orientations)} PAG orientations assigned ({PAG_ORI_MODE})')

## 6. Blocks and block orientations

Slices each packet into blocks.

- `BLOCK_THICKNESS_RANGE`: the smallest and largest block thickness, in voxels.
- `BLOCK_SLAB_CONNECTIVITY`: connectivity used when a slab is cut into connected pieces (6, 18 or 26).
- `KS_VARIANT_SELECTION`: how each block gets its Kurdjumov-Sachs variant. `'random_per_block'` picks randomly but avoids giving neighbouring blocks the same variant. `'deterministic'` always takes the variants in a fixed order.

In [ ]:
BLOCK_THICKNESS_RANGE = (2.0, 5.0)         # voxels
BLOCK_SLAB_CONNECTIVITY = 26
KS_VARIANT_SELECTION = 'random_per_block'  # or 'deterministic'
BLOCK_SEED = 42

fm_blk = steps_block_generation.generate_blocks(
    fm_pag, block_thickness_range=BLOCK_THICKNESS_RANGE, random_seed=BLOCK_SEED,
    block_slab_connectivity=BLOCK_SLAB_CONNECTIVITY)
fm_ori = steps_block_generation.assign_orientations(
    fm_blk, ks_variant_selection=KS_VARIANT_SELECTION, random_seed=BLOCK_SEED)
print(f'{len(fm_ori.all_blocks)} blocks, {len(fm_ori.block_orientations)} block orientations')

## 7. Sub-blocks (optional)

Off by default. When switched on, every block that is thick enough is cut into thinner sub-blocks.

- `SUBBLOCK_THICKNESS_RANGE_UM`: smallest and largest sub-block thickness, in the physical unit of the structure (microns here).
- `INTRABLOCK_ORI_SPREAD_DEG`: how much the orientations of the sub-blocks inside one block scatter, in degrees.
- `THIN_BLOCK_STRATEGY`: what to do with blocks too thin to divide. `'skip'` keeps the whole block as a single sub-block. `'upscale'` also keeps it whole, and expects you to have refined the grid yourself beforehand.

In [ ]:
GENERATE_SUBBLOCKS = False
SUBBLOCK_THICKNESS_RANGE_UM = (0.5, 1.5)
INTRABLOCK_ORI_SPREAD_DEG = 2.0
THIN_BLOCK_STRATEGY = 'skip'        # 'skip' or 'upscale'
SUBBLOCK_SEED = 42

if GENERATE_SUBBLOCKS:
    fm_sub = steps_subblock_generation.generate_subblocks(
        fm_ori, subblock_thickness_range_um=SUBBLOCK_THICKNESS_RANGE_UM,
        intrablock_ori_spread_deg=INTRABLOCK_ORI_SPREAD_DEG,
        thin_block_strategy=THIN_BLOCK_STRATEGY, random_seed=SUBBLOCK_SEED)
    print(f'{len(fm_sub.all_subblocks)} sub-blocks')
else:
    fm_sub = None
    print('Sub-block generation skipped.')

# The finished structure: this is what gets exported.
fm_final = fm_sub if GENERATE_SUBBLOCKS else fm_ori
LEVELS = ['pag', 'packet', 'block'] + (['subblock'] if GENERATE_SUBBLOCKS else [])

stages = steps_raw_export.PipelineStages(
    fm_base=fm_base, fm_with_pags=fm_pag, fm_with_blocks=fm_blk,
    fm_with_orientations=fm_ori, fm_with_subblocks=fm_sub)

## 8. Plots

Settings first. `VIZ_BACKEND = 'trame'` shows the 3D views inside the notebook so you can rotate them. Use `None` to open each one as a separate interactive view instead. `RANDOMIZE_IDS` shuffles the feature IDs before colouring, so neighbouring features don't end up with similar colours. `LEVELS_TO_PLOT` picks which levels of the hierarchy the plots below cover.

In [ ]:
VIZ_BACKEND = 'trame'            # 'trame' for in-notebook views, None for a separate view
RANDOMIZE_IDS = True
VIZ_CMAP = None                  # None builds a colour map that keeps features apart
VIZ_CMAP_SEED = 42
LEVELS_TO_PLOT = LEVELS          # any of 'pag', 'packet', 'block', 'subblock'

### 3D views

One interactive 3D view for each level of the hierarchy.

In [ ]:
for level in LEVELS_TO_PLOT:
    print(f'--- {level} ---')
    steps_visualization.render_3d(
        {'pag': fm_pag, 'packet': fm_blk, 'block': fm_ori, 'subblock': fm_sub}[level],
        level=level, jupyter_backend=VIZ_BACKEND, cmap=VIZ_CMAP, cmap_seed=VIZ_CMAP_SEED,
        randomize_ids=RANDOMIZE_IDS)

### Size and shape distributions

Histograms of feature sizes, aspect ratios and related measures, for each level. `MORPHOLOGY_BINS` sets the number of histogram bins.

In [ ]:
MORPHOLOGY_BINS = 20
MORPHOLOGY_FIGSIZE = (14, 7)

STATE_OF_LEVEL = {'pag': fm_pag, 'packet': fm_blk, 'block': fm_ori, 'subblock': fm_sub}

for level in LEVELS_TO_PLOT:
    collection = steps_visualization.distribution_stats(STATE_OF_LEVEL[level], level)
    aspect_ratios = steps_visualization.compute_aspect_ratios(STATE_OF_LEVEL[level], level)
    fig, _ = steps_visualization.plot_morphology_grid(
        collection, aspect_ratios, bins=MORPHOLOGY_BINS, figsize=MORPHOLOGY_FIGSIZE,
        suptitle=f'{level} morphology')
    report.add_image(fig, title=f'{level} morphology', section='Distributions')
    plt.show()

### Pole figures and misorientation distribution

Pole figures of the orientations at each level, and the distribution of misorientation angles between neighbouring features. `POLE_FAMILY` is the plane family, `'100'`, `'110'` or `'111'`. With `APPLY_SAMPLE_SYMMETRIES` on, every orientation is also copied through 180 degree rotations about the axes you switch on with `USE_RD`, `USE_TD` and `USE_ND`. The misorientation histogram runs over `MISORIENTATION_ANGLE_RANGE` degrees in `MISORIENTATION_N_BINS` bins.

In [ ]:
POLE_FAMILY = '100'
CRYSTAL_SYMMETRY = 'cubic'
APPLY_SAMPLE_SYMMETRIES = True
USE_RD, USE_TD, USE_ND = True, True, True
MISORIENTATION_N_BINS = 36
MISORIENTATION_ANGLE_RANGE = (0.0, 65.0)

for level in LEVELS_TO_PLOT:
    orientations = steps_visualization.get_level_orientations(stages, level)
    fig, ax = steps_visualization.plot_pole_figure(
        orientations, pole_family=POLE_FAMILY, crystal_symmetry=CRYSTAL_SYMMETRY,
        apply_sample_symmetries=APPLY_SAMPLE_SYMMETRIES, use_rd=USE_RD, use_td=USE_TD, use_nd=USE_ND)
    ax.set_title(f'{level} {{{POLE_FAMILY}}} pole figure')
    report.add_image(fig, title=f'{level} pole figure', section='Orientations')
    plt.show()

    collection = steps_visualization.distribution_stats(STATE_OF_LEVEL[level], level)
    mdf = steps_visualization.misorientation_distribution(
        orientations, collection, n_bins=MISORIENTATION_N_BINS,
        angle_range=MISORIENTATION_ANGLE_RANGE)
    if mdf is None:
        print(f'{level}: not enough neighbouring oriented features for a misorientation distribution.')
        continue
    fig, ax = plt.subplots(figsize=(6, 4))
    steps_visualization.plot_misorientation_distribution(
        mdf, ax=ax, title=f'{level} misorientation distribution')
    report.add_image(fig, title=f'{level} misorientation distribution', section='Orientations')
    plt.show()

### Slice maps

A flat cross-section through the structure, shown for the grains, the PAGs, the blocks and (if you generated them) the sub-blocks. `SLICE_AXIS` is the axis the slice is cut across, and `SLICE_INDEX` is the voxel layer (`None` takes the middle one). Colours are shuffled so neighbouring features are easy to tell apart.

In [ ]:
SLICE_AXIS = 'z'          # 'x', 'y' or 'z'
SLICE_INDEX = None        # None = middle layer
MAPS_SEED = 42

maps = build_visualization_data(
    fm_pag, fm_blk, fm_ori, fm_final, seed=MAPS_SEED, subblock_mode=GENERATE_SUBBLOCKS,
    domain_length=NX * VOXEL_SIZE)

grids = {'Grains': maps['grain_id_map'], 'PAGs': maps['pag_id_map'], 'Blocks': maps['block_fid_map']}
if GENERATE_SUBBLOCKS:
    grids['Sub-blocks'] = maps['subblock_fid_map']


def shuffled(grid, seed=0):
    values, inverse = np.unique(grid, return_inverse=True)
    order = np.random.default_rng(seed).permutation(len(values))
    return order[inverse].reshape(grid.shape)


axis = 'xyz'.index(SLICE_AXIS)
fig, axes = plt.subplots(1, len(grids), figsize=(4.2 * len(grids), 4.2))
for ax, (name, grid) in zip(np.atleast_1d(axes), grids.items()):
    k = grid.shape[axis] // 2 if SLICE_INDEX is None else SLICE_INDEX
    layer = np.take(shuffled(grid, MAPS_SEED), k, axis=axis)
    ax.imshow(layer.T, origin='lower', cmap='nipy_spectral', interpolation='nearest')
    ax.set_title(f'{name}, {SLICE_AXIS} = {k}')
    ax.set_xticks([]); ax.set_yticks([])
report.add_image(fig, title='Slice maps', section='Slices')
plt.show()

## 9. Abaqus deck

Writes the finished structure as an Abaqus `.inp` deck, using one element per voxel (or a tetrahedral split of each voxel). A new numbered folder is created for every export, so earlier decks are never overwritten.

- `ELEMENT_TYPE`: `'C3D8'` (linear hexahedra), `'C3D20'` (quadratic hexahedra), `'C3D4'` (linear tetrahedra, six per voxel) or `'C3D10'` (quadratic tetrahedra).
- `OUTPUT_UNIT`: the unit of the node coordinates, `'microns'`, `'mm'` or `'m'`.
- `EXPORT_LEVEL`: the hierarchy level that materials and sections are written at. `'auto'` uses the deepest level available. Otherwise pick `'pag'`, `'packet'`, `'block'` or `'subblock'`.
- Element sets are written for each hierarchy level with the prefixes in `ELSET_PREFIXES`. Switch on `ELSET_GRAIN_BOUNDARY`, `ELSET_JUNCTION_LINES` or `ELSET_JUNCTION_POINTS` to also get element sets of the voxels along grain boundaries, junction lines or junction points.
- `SCALE_FACTOR` rescales the exported grid. 2.0 doubles the voxels along each axis, and the deck gets that much bigger.

**What's in the deck.** The mesh (nodes and elements), the element sets, node sets on the faces of the domain, sections, and one material per feature. The materials are user-material stubs that carry each feature's Bunge Euler angles, so swap in your own constitutive model. There's also a default static step that holds the minus faces and pulls the z-plus face. The interactions file is only a placeholder: add periodic boundary conditions, ties or contact there if you need them. Treat the deck as a ready-made starting point, not a finished job.

In [ ]:
ELEMENT_TYPE = 'C3D8'            # 'C3D8', 'C3D20', 'C3D4' or 'C3D10'
OUTPUT_UNIT = 'microns'          # 'microns', 'mm' or 'm'
EXPORT_LEVEL = 'auto'            # 'auto', 'pag', 'packet', 'block' or 'subblock'
SCALE_FACTOR = 1.0
ELSET_PREFIXES = {'pag': 'es_pag_', 'pck': 'es_pck_', 'blk': 'es_blk_', 'subblk': 'es_subblk_'}
ELSET_GRAIN_BOUNDARY = False
ELSET_JUNCTION_LINES = False
ELSET_JUNCTION_POINTS = False
MESH_FOLDER_NAME = 'fm_steel_3d_voronoi'
CUSTOM_HEADER_MESSAGE = ''

options = {
    'export_level': EXPORT_LEVEL,
    **{f'prefix_{kind}': prefix for kind, prefix in ELSET_PREFIXES.items()},
    'elset_gb': ELSET_GRAIN_BOUNDARY,
    'elset_jl': ELSET_JUNCTION_LINES,
    'elset_jp': ELSET_JUNCTION_POINTS,
}

mesh_root = Path(OUTPUT_DIR) / 'FMSteel3D' / 'ABQInputFiles'
mesh_root.mkdir(parents=True, exist_ok=True)

deck_dir = export_deck(
    fm_final, ELEMENT_TYPE, mesh_root, folder_name=MESH_FOLDER_NAME, output_unit=OUTPUT_UNIT,
    scale_factor=SCALE_FACTOR, custom_message=CUSTOM_HEADER_MESSAGE, options=options,
    flat_output=False)
report.add_table([{'deck_folder': str(deck_dir), 'element_type': ELEMENT_TYPE}],
                 title='Abaqus deck', section='Export')
print('Deck written to:', deck_dir)
print(sorted(p.name for p in Path(deck_dir).iterdir()))

## 10. Raw arrays (optional)

Off by default. Saves the label grid and the orientations of each level as plain files you can load in your own scripts. `RAW_LEVELS = None` exports every level that exists; give a list to pick some. The file types are `'npy'`, `'pkl'` and `'npz'`.

In [ ]:
WRITE_RAW_EXPORT = False
RAW_FORMATS = ('npy', 'pkl', 'npz')
RAW_LEVELS = None

if WRITE_RAW_EXPORT:
    raw_result = steps_raw_export.export_raw(
        stages, out_dir=Path(OUTPUT_DIR) / 'FMSteel3D' / 'RawExports' / 'fm_steel_3d_voronoi',
        formats=RAW_FORMATS, levels=RAW_LEVELS)
    print(raw_result['out_dir'], raw_result['levels_exported'])
else:
    print('Raw export skipped.')

## Finish

Writes everything collected in the report to a single `report.html`.

In [ ]:
report_path = steps_start.finish_report(report)
print('Report written to:', report_path)